# Instruction Fine-Tuning LLM using SFT for Financial Sentiment: A Step-by-Step Guide

Run cells in order. This uses the full official OPT-1.3B checkpoint and a hash-verified 64/16 subset of the original 20,000/2,000 public Deep Lake splits for twenty real local LoRA updates—not the original ten-epoch experiment. It needs several GB of free RAM/disk and uses MPS float16 base weights on the tested Apple host or CPU float32 otherwise. Adapter training uses float32 trainable weights; merge and saved-model reload run on CPU float32 for stable arithmetic. No rented GPU, W&B upload or Hub push. Keep actual generations, including wrong or repetitive text, as they are.

```bash
python -m pip install torch==2.10.0 transformers==5.22.0 peft==0.21.0 trl==1.17.1 datasets==5.1.0 deeplake==3.9.52 huggingface-hub==2.4.0 numpy==2.4.6
```

In [ ]:
import os, json, hashlib, tempfile, gc, math, random
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from huggingface_hub import snapshot_download
set_seed(42)
torch.set_num_threads(2)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
dtype = torch.float16 if device.type == 'mps' else torch.float32
work = tempfile.TemporaryDirectory(prefix='tdb-financial-sft-')
work_path = Path(work.name)
print('Device:', device, 'Base dtype:', dtype)
print({p: version(p) for p in ['torch', 'transformers', 'trl', 'peft', 'deeplake']})

In [ ]:
import deeplake
SELECTION = {'train': {'path': 'hub://genai360/FingGPT-sentiment-train-set', 'totalRows': 20000, 'indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63], 'sha256': 'ec30ca747fab91eb75427a55719b3eb28687fb1130969c94cea74705f64bd2c7'}, 'valid': {'path': 'hub://genai360/FingGPT-sentiment-valid-set', 'totalRows': 2000, 'indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], 'sha256': 'b16dbd25c816e8767ff9f74d63e50d5fda0d418bca91f9594106991819e7c145'}}
def read_selected(spec):
    dataset = deeplake.load(spec['path'], read_only=True, verbose=False,
                            memory_cache_size=64, local_cache_size=0)
    if len(dataset) != spec['totalRows']:
        raise RuntimeError('Original dataset row count changed')
    rows = [{k: str(dataset[i][k].text()) for k in ['input', 'instruction', 'output']}
            for i in spec['indices']]
    encoded = json.dumps(rows, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()
    if hashlib.sha256(encoded).hexdigest() != spec['sha256']:
        raise RuntimeError('Selected original dataset bytes changed')
    return rows
train_rows = read_selected(SELECTION['train'])
valid_rows = read_selected(SELECTION['valid'])
normalize = lambda text: ' '.join(text.split()).casefold()
train_inputs = {normalize(row['input']) for row in train_rows}
valid_inputs = {normalize(row['input']) for row in valid_rows}
assert len(train_inputs) == 64 and len(valid_inputs) == 16 and not train_inputs & valid_inputs
allowed_labels = {'negative', 'neutral', 'positive', 'strong negative', 'strong positive',
                  'moderately negative', 'moderately positive', 'mildly negative', 'mildly positive'}
assert all(row['output'] in allowed_labels and row['output'] in row['instruction'].lower()
           for row in train_rows + valid_rows)
print('Original train/validation rows:', SELECTION['train']['totalRows'], SELECTION['valid']['totalRows'])
print('Selected disjoint training/validation rows:', len(train_rows), len(valid_rows))
print('Selected labels:', sorted({row['output'] for row in train_rows + valid_rows}))

In [ ]:
def prepare_sample_text(example):
    prompt = example['instruction'] + '\n\nContent: ' + example['input'] + '\n\nSentiment:'
    return {'prompt': prompt, 'completion': ' ' + example['output']}
train_dataset = Dataset.from_list([prepare_sample_text(row) for row in train_rows])
eval_dataset = Dataset.from_list([prepare_sample_text(row) for row in valid_rows])
print(train_dataset[0]['prompt'] + train_dataset[0]['completion'])
print('Prompt/completion columns:', train_dataset.column_names)

In [ ]:
MODEL_ID = 'facebook/opt-1.3b'
REVISION = '3f5c25d0bc631cb57ac65913f76e22c2dfb61d62'
WEIGHT_SHA256 = 'cf7d5c970d6ddbd3b03009b397c0422e147edd5c8020d47a8d2fac0b11a3b08d'
cache_dir = Path(tempfile.gettempdir()) / 'tdb-archive27-model-cache'
checkpoint = Path(snapshot_download(MODEL_ID, revision=REVISION, token=False,
    cache_dir=cache_dir, allow_patterns=['*.json', '*.txt', 'vocab.json', 'merges.txt', 'pytorch_model.bin']))
with (checkpoint / 'pytorch_model.bin').open('rb') as handle:
    assert hashlib.file_digest(handle, 'sha256').hexdigest() == WEIGHT_SHA256
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
lengths = [len(tokenizer(row['prompt'] + row['completion'] + tokenizer.eos_token)['input_ids'])
           for row in list(train_dataset) + list(eval_dataset)]
assert max(lengths) <= 256, 'Do not silently truncate sentiment labels'
print('Tokenizer vocabulary:', len(tokenizer), 'Maximum complete example length:', max(lengths))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(checkpoint, local_files_only=True,
    dtype=dtype, device_map={'': str(device)}, attn_implementation='eager')
model.config.use_cache = False
query_text = ('What is the sentiment of this news? Please choose an answer from '
              '{strong negative/moderately negative/mildly negative/neutral/mildly positive/moderately positive/strong positive}.\n\nContent: UPDATE 1-AstraZeneca sells rare '
              'cancer drug to Sanofi for up to $300 mln.\n\nSentiment:')
assert not any('astrazeneca sells rare cancer drug' in normalize(row['input'])
               for row in train_rows + valid_rows)
inference_inputs = tokenizer(query_text, return_tensors='pt').to(device)
def generate_answer(current_model):
    current_model.eval()
    with torch.inference_mode():
        current_inputs = {k: value.to(current_model.device) for k, value in inference_inputs.items()}
        generated = current_model.generate(**current_inputs, max_new_tokens=40,
            do_sample=False, num_beams=1, repetition_penalty=1.5,
            pad_token_id=tokenizer.pad_token_id, use_cache=True)
    return tokenizer.decode(generated[0, inference_inputs.input_ids.shape[1]:], skip_special_tokens=True)
base_answer = generate_answer(model)
print('Base-model continuation:', repr(base_answer))
print('Base parameter count:', sum(p.numel() for p in model.parameters()))

In [ ]:
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'], bias='none', task_type='CAUSAL_LM')
training_args = SFTConfig(output_dir=str(work_path / 'training'), max_steps=20,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=1, learning_rate=1e-4, lr_scheduler_type='cosine',
    warmup_steps=2, weight_decay=0.05, optim='adamw_torch',
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=False, bf16=False, use_cpu=device.type == 'cpu',
    eval_strategy='no', save_strategy='no', logging_steps=5,
    report_to='none', push_to_hub=False, dataloader_num_workers=0,
    dataloader_pin_memory=False, seed=42, data_seed=42,
    packing=False, max_length=256, completion_only_loss=True)
print('20 optimizer steps; batch 1; completion-only loss; no packing or hosted reporting.')
print('The base uses the explicit dtype above, not automatic mixed precision.')

In [ ]:
gradient_checks = []
class GradientCheck(TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        assert grads and all(torch.isfinite(g).all().item() for g in grads)
        assert any(torch.count_nonzero(g).item() > 0 for g in grads)
        gradient_checks.append(state.global_step)
        return control
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=eval_dataset, processing_class=tokenizer, peft_config=lora_config,
    callbacks=[GradientCheck()])
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
assert trainable == 3145728
assert all('lora_' in name for name, p in trainer.model.named_parameters() if p.requires_grad)
def frozen_hash(current_model):
    digest = hashlib.sha256()
    for name, parameter in current_model.named_parameters():
        if not parameter.requires_grad:
            digest.update(name.encode())
            digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()
frozen_before = frozen_hash(trainer.model)
print('Trainable:', trainable, 'Total:', total, 'Trainable percent:', 100 * trainable / total)
probe = trainer.data_collator([trainer.train_dataset[0]])
assert (probe['labels'] == -100).any() and (probe['labels'] != -100).any()
assert torch.equal(probe['labels'][probe['labels'] != -100], probe['input_ids'][probe['labels'] != -100])
print('Completion labels:', tokenizer.decode(probe['labels'][probe['labels'] != -100]))
before_eval = trainer.evaluate()
print('Validation loss before training:', before_eval['eval_loss'])

In [ ]:
train_result = trainer.train()
assert trainer.state.global_step == 20 and len(gradient_checks) == 20
assert frozen_hash(trainer.model) == frozen_before
after_eval = trainer.evaluate()
assert math.isfinite(after_eval['eval_loss']) and math.isfinite(train_result.training_loss)
print('Actual optimizer updates:', trainer.state.global_step, 'Finite nonzero gradient checks:', len(gradient_checks))
print('Validation loss before/after:', before_eval['eval_loss'], after_eval['eval_loss'])
print('Frozen base parameter hash unchanged:', frozen_before)

In [ ]:
trainer.model.eval()
probe_inputs = {k: value.to(device) for k, value in probe.items() if k != 'labels'}
with torch.inference_mode():
    logits = trainer.model(**probe_inputs).logits.detach().cpu().float()
    loss = trainer.model(**{k: value.to(device) for k, value in probe.items()}).loss.detach().cpu().float()
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = probe['labels'][:, 1:].contiguous()
independent_loss = torch.nn.functional.cross_entropy(shift_logits.reshape(-1, shift_logits.shape[-1]),
    shift_labels.reshape(-1), ignore_index=-100)
torch.testing.assert_close(loss, independent_loss, atol=0.01, rtol=0.005)
adapter_path = work_path / 'adapter'
trainer.model.save_pretrained(adapter_path, safe_serialization=True)
tokenizer.save_pretrained(adapter_path)
base = trainer.model.unload()
reloaded = PeftModel.from_pretrained(base, adapter_path, is_trainable=False)
reloaded.eval()
with torch.inference_mode():
    reloaded_logits = reloaded(**probe_inputs).logits.detach().cpu().float()
torch.testing.assert_close(logits, reloaded_logits, atol=0.002, rtol=0.002)
print('Independent completion cross-entropy:', float(independent_loss))
print('Adapter save/reload maximum logit difference:', float((logits - reloaded_logits).abs().max()))

In [ ]:
# Merge and verify in float32: half-precision in-place weight addition drifts.
reloaded = reloaded.to(device='cpu', dtype=torch.float32).eval()
cpu_probe_inputs = {k: value.cpu() for k, value in probe_inputs.items()}
with torch.inference_mode():
    reference_float32_logits = reloaded(**cpu_probe_inputs).logits.detach().float()
merged = reloaded.merge_and_unload(safe_merge=True)
merged.eval()
with torch.inference_mode():
    merged_logits = merged(**cpu_probe_inputs).logits.detach().float()
merge_error = float((merged_logits - reference_float32_logits).abs().max())
torch.testing.assert_close(merged_logits, reference_float32_logits, atol=0.0002, rtol=0.0002)
merged_path = work_path / 'merged'
merged.save_pretrained(merged_path, safe_serialization=True)
tokenizer.save_pretrained(merged_path)
print('Merge maximum logit difference:', merge_error)
print('Merged model saved with its matching tokenizer.')
del trainer, model, base, reloaded, merged
gc.collect()
if device.type == 'mps':
    torch.mps.empty_cache()
final_model = AutoModelForCausalLM.from_pretrained(merged_path, local_files_only=True,
    dtype=torch.float32, device_map={'': 'cpu'}, attn_implementation='eager')
final_model.eval()
with torch.inference_mode():
    saved_logits = final_model(**cpu_probe_inputs).logits.detach().float()
torch.testing.assert_close(saved_logits, merged_logits, atol=0.002, rtol=0.002)
print('Saved merged-model reload maximum logit difference:', float((saved_logits - merged_logits).abs().max()))

In [ ]:
# Use the same tested MPS/float16 generation path as the base continuation.
# Float32 CPU merge/reload parity has already been checked above.
if device.type == 'mps':
    final_model = final_model.to(device=device, dtype=torch.float16)
tuned_answer = generate_answer(final_model)
print('Input:', query_text)
print('Base continuation:', repr(base_answer))
print('Fine-tuned continuation:', repr(tuned_answer))
print('The continuations are actual outputs from a bounded 20-step run, not fabricated sentiment labels.')

In [ ]:
assert train_inputs.isdisjoint(valid_inputs)
assert trainable == 3145728 and len(gradient_checks) == 20
assert torch.isfinite(saved_logits).all()
assert not any('lora_' in name for name, _ in final_model.named_parameters())
print('PASS: original real data hashes, disjoint inputs, completion labels/loss, twenty real updates, finite gradients, frozen base and adapter/merge/reload checks.')
del final_model
gc.collect()
if device.type == 'mps':
    torch.mps.empty_cache()
work.cleanup()
print('Owned training/checkpoint directory removed; reusable official download cache retained.')

## Original source-code and transcript appendix

Nonexecuting original archive material; use the maintained cells above.

```text
!pip install -q transformers==4.32.0 
deeplake==3.6.19 
trl==0.6.0
peft==0.5.0 
wandb==0.15.8
```

```text
import deeplake

# Connect to the training and testing datasets
ds = deeplake.load('hub://genai360/FingGPT-sentiment-train-set')
ds_valid = deeplake.load('hub://genai360/FingGPT-sentiment-valid-set')
ds
```

```text
def prepare_sample_text(example):
    """Prepare the text from a sample of the dataset."""
    text = f"{example['instruction'].text()}\n\nContent: {example['input'].text()}\n\nSentiment: {example['output'].text()}"
    return text
```

```text
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")
```

```text
from trl.trainer import ConstantLengthDataset

train_dataset = ConstantLengthDataset(
    tokenizer,
    ds,
    formatting_func=prepare_sample_text,
    infinite=True,
    seq_length=1024
)
```

```text
iterator = iter( train_dataset )
sample = next( iterator )
print( sample )
```

```text
eval_dataset = ConstantLengthDataset(
    tokenizer,
    ds_valid,
    formatting_func=prepare_sample_text,
    seq_length=1024
)
```

```text
from transformers import TrainingArguments

# Define LoRAConfig
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
```

```text
training_args = TrainingArguments(
    output_dir="./OPT-fine_tuned-FinGPT-CPU",
    dataloader_drop_last=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    logging_steps=5,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    weight_decay=0.05,
    ddp_find_unused_parameters=False,
    run_name="OPT-fine_tuned-FinGPT-CPU",
    report_to="wandb",
)
```

```text
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-1.3b", torch_dtype=torch.bfloat16
)
```

```text
from torch import nn

for param in model.parameters():
  param.requires_grad = False  # freeze the model - train adapters later
  if param.ndim == 1:
    # cast the small parameters (e.g. layernorm) to fp32 for stability
    param.data = param.data.to(torch.float32)

model.gradient_checkpointing_enable()  # reduce number of stored activations
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
  def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)
```

```text
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    packing=True,
)
```

```text
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

print_trainable_parameters(trainer.model)
```

```text
print("Training...")
trainer.train()
```

```text
# Load the base model (OPT-1.3B)
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
  "facebook/opt-1.3b", return_dict=True, torch_dtype=torch.bfloat16
)

# Load the LoRA adaptors
from peft import PeftModel

# Load the Lora model
model = PeftModel.from_pretrained(model, "./OPT-fine_tuned-FinGPT-CPU/<desired_checkpoint>/")
model.eval()
model = model.merge_and_unload()

# Save for future use
model.save_pretrained("./OPT-fine_tuned-FinGPT-CPU/merged")
```

```text
inputs = inputs_tokenizer("What is the sentiment of this news? Please choose an answer from {strong negative/moderately negative/mildly negative/neutral/mildly positive/moderately positive/strong positive}, then provide some short reasons.\n\nContent: UPDATE 1-AstraZeneca sells rare cancer drug to Sanofi for up to $300 mln.\n\nSentiment: ", 
    return_tensors="pt").to("cuda:0")
```

```text
generation_output_1 = model.generate(**inputs, 
                                  return_dict_in_generate=True, 
                                  output_scores=True, 
                                  max_length=256, 
                                  num_beams=1, 
                                  do_sample=True, 
                                  repetition_penalty=1.5, 
                                  length penalty=2.)

tokenizer.decode(generation_outputl['sequences'][0]
```

```text
generation_output_2 = model2.generate(**inputs, 
                                      return_dict_in_generate=True, 
                                      output_scores=True, 
                                      max_length=256, 
                                      num beams=1, 
                                      do_sample=False, 
                                      repetition_penalty=1.5, 
                                      length_penalty=2)

tokenizer.decode(generation_output2['sequences'][0])
```

```text
Dataset(path=’hub://genai360/FingGPT-sentiment-valid-set’, read_only=True, tensors=[‘input’, ‘instruction’, ‘output’])
```

```text
What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
```

```text
Content: Diageo Shares Surge on Report of Possible Takeover by Lemann
```

```text
Sentiment: positive
```

```text
{‘input_ids’: tensor([50118, 35212, 8913, …, 2430, 2, 2]),’labels’: tensor([50118, 35212, 8913, …, 2430, 2, 2])
```

```text
trainable params: 3145728 || all params: 1318903808 || trainable%: 0.23851079820371554
```

```text
</s>What is the sentiment of this news? Please choose an answer from (negative/neutral/positive}\n\nn\nContent: UPDATE 1-AstraZeneca sells rare cancer drug to S anofi for up to $300 mln.\n\nSentiment:?????????????????(0 votes average) (100% rating, 12 comments) Rate Article Share on FacebookTweetThis one looks good an d they have a product that can be used in combination with chemo so maybe it will help their patients get rid if these cancers as well as helping them live longe r. This doesn’t sound like some crap made by Astra at first glance. What do ya’ll think about what was announced today though? Do you agree or disagree with thes e statements??? Also tell us your thoughts below:-) Thanks! Ravi Posted 06.04 2014 1433 words 3+ Comments Upvote 10 downvotes Content Rating 100% Vote [Articl e] More content… View vote results page Popularity Ratio 0.0075962 Shares User Login Signup Google Plus Email Password Not logged into LinkedIn Join now Co mment! A survey just came out compar…’
```

```text
</s>What is the sentiment of this news? Please choose an answer from (negative/neutral/positive}\n\nn\nContent: UPDATE 1-AstraZeneca sells rare cancer drug to S anofi for up to $300 mln.\n\nSentiment: positive</s>
```